In [1]:
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import os
import liana as li
from liana.method import cellphonedb, cellchat
from tqdm import tqdm
#liana dotplot returns a ggsave object...
from plotnine import ggsave, ggplot
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [3]:
!pwd

/misc/s/medinfo/bmi776/bmi776-s25/project_files/project_siwei/Submit/Code/AverageExpressionLevels


In [4]:
folder = "../../ProcessedData/IndividualMiceData/"
Mice = ["GF_HF_6B","GF_HF_13L","SPF_HF_118","SPF_HF_131","SPF_HF_133","GF_HFVHC_5A","GF_HFVHC_16L","SPF_HFVHC_124","SPF_HFVHC_136","SPF_HFVHC_137"]
method = ["CellPhoneDB", "CellChat"]
def get_data():
    data = {}
    for mouse in tqdm(Mice):
        data[mouse] = {}
        for m in method:
            file = folder+mouse+"_"+m+".h5ad"
            data[mouse][m] = sc.read_h5ad(file)
    return data
data = get_data()

100%|██████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:21<00:00,  2.15s/it]


In [5]:
mouse_curr = list(data.keys())[0]
mouse_curr_data = {}
for m in method:
    x = data[mouse_curr][m].uns["cpdb_res"]
    if m == 'CellPhoneDB':
        # x = x[x['lr_means'] != 0]
        x = x[x['cellphone_pvals'] < 0.05]
        x['Ligand_Value'] = x['ligand_means']
        x['Receptor_Value'] = x['receptor_means']
    else:
        # x = x[x['lr_probs'] != 0]
        x = x[x['cellchat_pvals'] < 0.05]
        x['Ligand_Value'] = x['ligand_trimean']
        x['Receptor_Value'] = x['receptor_trimean']
    x = x[['ligand', 'receptor', 'source', 'target', 'Ligand_Value', 'Receptor_Value']]
    x['Source -> Target'] = x['source'] + ' -> ' + x['target']
    x['Ligand -> Receptor'] = x['ligand'] + ' -> ' + x['receptor']
    x['Ligand_Receptor_Difference'] = abs(x['Ligand_Value'] - x['Receptor_Value'])
    mouse_curr_data[m] = x[['Source -> Target', 'Ligand -> Receptor','Ligand_Receptor_Difference']]

df1 = mouse_curr_data[method[0]]
df2 = mouse_curr_data[method[1]]
    
# Create a combined key for comparison in both DataFrames
df1['combined_key'] = df1['Source -> Target'].astype(str) + '||' + df1['Ligand -> Receptor'].astype(str)
df2['combined_key'] = df2['Source -> Target'].astype(str) + '||' + df2['Ligand -> Receptor'].astype(str)

# Identify the common rows based on the combined key
common_keys = pd.merge(df1['combined_key'], df2['combined_key'], on='combined_key', how='inner')['combined_key']

# Remove the common rows from df1
df1_filtered = df1[~df1['combined_key'].isin(common_keys)].drop(columns=['combined_key'])

# Remove the common rows from df2
df2_filtered = df2[~df2['combined_key'].isin(common_keys)].drop(columns=['combined_key'])

print(df1_filtered.shape[0],df2_filtered.shape[0])

16220 3187


In [6]:
df_plot = merged_df.copy()

df_plot['Interaction'] = df_plot['Source -> Target'] + ' | ' + df_plot['Ligand -> Receptor']

# --- Plot for CellPhoneDB values ---
# Melt the DataFrame to long format for CellPhoneDB
df_cellphonedb_melted = pd.melt(
    df_plot,
    id_vars=['Interaction'],
    value_vars=['Ligand_Value_CellPhoneDB', 'Receptor_Value_CellPhoneDB'],
    var_name='Value_Type',
    value_name='Value'
)

# Create the bar plot for CellPhoneDB
plt.figure(figsize=(16, 6))  # Increased figure size for better readability
sns.barplot(
    data=df_cellphonedb_melted,
    x='Interaction',
    y='Value',
    hue='Value_Type',
    dodge=True
)
plt.xlabel('Source -> Target | Ligand -> Receptor')
plt.ylabel('Value')
plt.title('CellPhoneDB Ligand and Receptor Values')
plt.xticks(rotation=90, fontsize=8)  # Rotated and smaller x-axis labels
plt.tight_layout()
plt.savefig("figures/CellPhoneDB_Mouse1_lr_values.png")
# plt.show()

# --- Plot for CellChat values ---
# Melt the DataFrame to long format for CellChat
df_cellchat_melted = pd.melt(
    df_plot,
    id_vars=['Interaction'],
    value_vars=['Ligand_Value_CellChat', 'Receptor_Value_CellChat'],
    var_name='Value_Type',
    value_name='Value'
)

# Create the bar plot for CellChat
plt.figure(figsize=(16, 6))  # Increased figure size for better readability
sns.barplot(
    data=df_cellchat_melted,
    x='Interaction',
    y='Value',
    hue='Value_Type',
    dodge=True
)
plt.xlabel('Source -> Target | Ligand -> Receptor')
plt.ylabel('Value')
plt.title('CellChat Ligand and Receptor Values')
plt.xticks(rotation=90, fontsize=8)  # Rotated and smaller x-axis labels
plt.tight_layout()
plt.savefig("figures/CellChat_Mouse1_lr_values.png")
# plt.show()

NameError: name 'merged_df' is not defined